# Lagrangian Neural Network — Simple Pendulum

This notebook introduces **Lagrangian Neural Networks (LNNs)** and compares them to
a plain VanillaNN on the simple pendulum — a strongly nonlinear, conservative system.

**Core idea:** Instead of learning the acceleration $(q, \dot{q}) \mapsto \ddot{q}$
directly, the LNN learns a scalar **Lagrangian** $L_\theta(q, \dot{q}) \in \mathbb{R}$
and derives the equations of motion analytically via the Euler-Lagrange equations:

$$\hat{\ddot{q}} = \left(\nabla_{\dot{q}\dot{q}}^2 L_\theta\right)^{-1}
\left[\nabla_q L_\theta - \left(\nabla_{q\dot{q}}^2 L_\theta\right)\dot{q}\right]$$

All gradients are computed by `autograd` — the Lagrangian structure is encoded in the
*architecture*, not in any loss penalty.

**Key insight:** With only 20 data points from a near-separatrix trajectory (highly
nonlinear), the LNN stays on the correct energy shell far beyond the training window.
Energy conservation is a mathematical consequence of deriving dynamics from a
Lagrangian — not a soft constraint.


In [ ]:
import os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))
from systems import SimplePendulum
from models import LNN, VanillaNN
from utils.training import train_dynamics_model

SEED = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"Device: {DEVICE}")

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
COLORS = {"True": "#2c3e50", "LNN": "#3498db", "VanillaNN": "#e74c3c"}

## System and Data

**Simple pendulum:**  $\ddot{\theta} = -(g/L)\sin\theta$  with $g=9.81$, $L=1$.

The initial condition $\theta_0 = 2.5\,\text{rad}$ is chosen deliberately near the
**separatrix** — the boundary between oscillatory and rotational motion. At this
amplitude, $\sin\theta \approx \theta$ is a poor approximation and the trajectory
is strongly nonlinear.

Both models are trained on **only 20 samples** of $(q, \dot{q}, \ddot{q})$ from
$t \in [0, 3\,\text{s}]$ and evaluated by integrating to $t = 12\,\text{s}$
(4× the training horizon).

Both models see *identical training data* — the difference is purely structural.


In [ ]:
pend = SimplePendulum(g=9.81, length=1.0)
THETA0, TDOT0 = 2.5, 0.0
T_TRAIN, T_EVAL = (0.0, 3.0), (0.0, 12.0)
N_TRAIN, N_EVAL = 20, 1200

traj_train = pend.rollout(THETA0, TDOT0, T_TRAIN, N_TRAIN)
traj_eval  = pend.rollout(THETA0, TDOT0, T_EVAL,  N_EVAL)

train_ds = {"theta":      traj_train["theta"],
            "theta_dot":  traj_train["theta_dot"],
            "theta_ddot": traj_train["theta_ddot"]}

t_eval  = traj_eval["t"]
th_true = traj_eval["theta"]
td_true = traj_eval["theta_dot"]
E_true  = pend.total_energy(th_true, td_true)

plt.figure(figsize=(10, 3))
plt.plot(t_eval, th_true, color=COLORS["True"], lw=1.5, ls="--", label="Ground truth θ(t)")
plt.scatter(traj_train["t"], traj_train["theta"], s=40, color=COLORS["True"], zorder=5, label="20 training pts")
plt.axvline(T_TRAIN[1], color="gray", ls=":", lw=1.2, label="train end")
plt.axvspan(T_TRAIN[1], T_EVAL[1], alpha=0.06, color="gray")
plt.xlabel("t (s)"); plt.ylabel("θ (rad)"); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Models and Training

Both models have **3 hidden layers of 64 units** and are trained to minimise MSE on
the acceleration:

$$\mathcal{L} = \frac{1}{N}\sum_{i=1}^N \|\hat{\ddot{q}}_i - \ddot{q}_i\|^2$$

The loss is *identical* for both. The difference:
- **VanillaNN**: a standard MLP that maps $(q, \dot{q}) \mapsto \ddot{q}$ directly
- **LNN**: a scalar MLP that maps $(q, \dot{q}) \mapsto L_\theta$, then derives
  $\ddot{q}$ via the Euler-Lagrange equation using `autograd`

The LNN forward pass requires inverting the $1\times1$ mass matrix (trivial here)
and computing second derivatives of the network — `autograd` handles both automatically.


In [ ]:
EPOCHS = 2_000

models = {
    "LNN":       LNN(      n_dof=1, hidden_dim=64, n_layers=3),
    "VanillaNN": VanillaNN(n_dof=1, hidden_dim=64, n_layers=3),
}
for name, model in models.items():
    print(f"\nTraining {name} ...")
    hist = train_dynamics_model(model, train_ds, n_epochs=EPOCHS,
                                batch_size=16, device=DEVICE, verbose=True, log_every=500)
    print(f"  final loss={hist['train_loss'][-1]:.4e}  t={hist['wall_time']:.1f}s")

## RK4 Rollout and Evaluation

After training, we integrate both models forward using a **4th-order Runge-Kutta**
integrator from the same initial conditions. This is the standard way to evaluate
dynamics models: the model predicts the acceleration at each step; the integrator
propagates the state.

Errors accumulate over time — any small systematic bias in the predicted acceleration
will drift the trajectory. The LNN is immune to one class of drift: energy drift.
Because it derives accelerations from a Lagrangian, its trajectories conserve the
learned Hamiltonian exactly.


In [ ]:
def rk4(model, theta0, thetadot0, t_array, device="cpu"):
    q, qd = float(theta0), float(thetadot0)
    qs, qds = [q], [qd]
    def acc(qi, qdi):
        qt  = torch.tensor([[qi]],  dtype=torch.float32, device=device)
        qdt = torch.tensor([[qdi]], dtype=torch.float32, device=device)
        with torch.enable_grad():
            return model.predict_acceleration(qt, qdt).detach().cpu().item()
    for dt in np.diff(t_array):
        k1v = qd;              k1a = acc(q,              qd)
        k2v = qd + .5*dt*k1a; k2a = acc(q + .5*dt*k1v, qd + .5*dt*k1a)
        k3v = qd + .5*dt*k2a; k3a = acc(q + .5*dt*k2v, qd + .5*dt*k2a)
        k4v = qd +    dt*k3a; k4a = acc(q +    dt*k3v, qd +    dt*k3a)
        q  = q  + (dt/6)*(k1v + 2*k2v + 2*k3v + k4v)
        qd = qd + (dt/6)*(k1a + 2*k2a + 2*k3a + k4a)
        qs.append(q); qds.append(qd)
    return np.array(qs), np.array(qds)

split = T_TRAIN[1]
in_m, out_m = t_eval <= split, t_eval > split

preds = {}
for name, model in models.items():
    model.eval()
    th, td = rk4(model, THETA0, TDOT0, t_eval, DEVICE)
    E_pred   = pend.total_energy(th, td)
    rmse_in  = float(np.sqrt(np.mean((th[in_m]  - th_true[in_m])**2)))
    rmse_out = float(np.sqrt(np.mean((th[out_m] - th_true[out_m])**2)))
    e_drift  = float(np.mean(np.abs(E_pred - E_true[0]) / (np.abs(E_true[0]) + 1e-8)))
    preds[name] = {"theta": th, "E": E_pred, "interp": rmse_in, "extrap": rmse_out, "edrift": e_drift}
    print(f"{name:<12}  interp={rmse_in:.4f}  extrap={rmse_out:.4f}  E-drift={e_drift:.4f}")

gain = preds["VanillaNN"]["extrap"] / (preds["LNN"]["extrap"] + 1e-12)
print(f"\nLNN extrap improvement over VanillaNN: {gain:.1f}×")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
fig.suptitle(f"LNN vs VanillaNN — Simple Pendulum  (θ₀=2.5 rad, near-separatrix)\n"
             f"{N_TRAIN} training pts from [0, 3 s]  |  shaded = extrapolation",
             fontsize=12, fontweight="bold")

ax1.plot(t_eval, th_true, color=COLORS["True"], lw=1, ls="--", label="Ground truth", zorder=5)
for name in ["LNN", "VanillaNN"]:
    p = preds[name]
    ax1.plot(t_eval, np.clip(p["theta"], -15, 15), color=COLORS[name], lw=1.8,
             label=f"{name}  (extrap={p['extrap']:.4f})")
ax1.axvline(split, color="gray", ls=":", lw=1.4, label="train end")
ax1.axvspan(split, t_eval[-1], alpha=0.07, color="gray")
ax1.set_ylabel("θ (rad)", fontsize=11); ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)
ax1.set_title("Trajectory rollout", fontsize=11, fontweight="bold")

for name in ["LNN", "VanillaNN"]:
    err = np.abs(np.clip(preds[name]["theta"], -50, 50) - th_true)
    ax2.semilogy(t_eval, err + 1e-6, color=COLORS[name], lw=1.6, label=name)
ax2.axvline(split, color="gray", ls=":", lw=1.4)
ax2.axvspan(split, t_eval[-1], alpha=0.07, color="gray")
ax2.set_ylabel("|error|  (log scale)", fontsize=11); ax2.set_xlabel("Time (s)", fontsize=11)
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "lnn_vs_vanilla.png"), dpi=150, bbox_inches="tight")
plt.show()